## Initial EDA

In [4]:
# import libraries

import pandas as pd
import ast
import numpy as np
from shapely import Point, Polygon

In [5]:
# File Path to processed shot table

processed_output_folder = 'C:/Users/bquev/Desktop/6311 Capstone/output/processed_data/'

In [6]:
# Load the shots table from initial statsbomb load data

all_shots_df = pd.read_csv(processed_output_folder + 'shots.csv')

print(all_shots_df.shape)

(66854, 159)


In [7]:
# Review match and team coverage for each competition season

coverage_rows = []

for (competition, season), seasons_shots_df in all_shots_df.groupby(['competition_name', 'season_name']):

    # Count matches and teams per season

    matches = seasons_shots_df['match_id'].nunique()
    teams = seasons_shots_df['team_id'].nunique()

    # Find how many matches each team appears in, then keep the highest
    # If one team appears in every match, the season follows that team only

    team_matches = seasons_shots_df.groupby('team_id')['match_id'].nunique()
    most_team_matches = team_matches.max()

    # Label how much of the season the data covers

    if matches == 1:
        coverage = 'Single match'
    elif most_team_matches == matches:
        coverage = 'One team in every match'
    else:
        coverage = 'Multiple teams'

    # Save results for each competition season

    coverage_rows.append({'competition': competition,
                          'season': season,
                          'matches': matches,
                          'teams': teams,
                          'season_coverage': coverage})

# Create a single table, sorted in descending order based on matches

season_coverage_df = pd.DataFrame(coverage_rows)
season_coverage_df = season_coverage_df.sort_values('matches', ascending=False)

print(season_coverage_df.head(15))

               competition     season  matches  teams          season_coverage
47                 La Liga  2015/2016      380     20           Multiple teams
61          Premier League  2015/2016      380     20           Multiple teams
63                 Serie A  2015/2016      380     20           Multiple teams
55                 Ligue 1  2015/2016      377     20           Multiple teams
34     Indian Super league  2021/2022      115     11           Multiple teams
33          FIFA World Cup       2022       64     32           Multiple teams
32          FIFA World Cup       2018       64     32           Multiple teams
2   African Cup of Nations       2023       52     24           Multiple teams
64               UEFA Euro       2020       51     24           Multiple teams
65               UEFA Euro       2024       51     24           Multiple teams
60          Premier League  2003/2004       38     20  One team in every match
46                 La Liga  2014/2015       38     2

## Coverage of data

Based on these results, the 2015/2016 season is the only season with complete or nearly complete coverage across the four European leagues. Ligue 1 is missing three matches, but this is unlikely to significantly affect the overall shot analysis.

In [8]:
# Filter to the four European leagues for the 2015/16 season
# Premier League (2), Ligue 1 (7), La Liga (11), Serie A (12), 2015/16 season (27)

major_4_competitions_shots_df = all_shots_df.loc[
    all_shots_df['competition_id'].isin([2, 7, 11, 12])
    & (all_shots_df['season_id'] == 27)
].copy()

# Check rows and columns

print(major_4_competitions_shots_df.shape)

# Check all matches came through

print(major_4_competitions_shots_df.groupby('competition_name')['match_id'].nunique())


(37888, 159)
competition_name
La Liga           380
Ligue 1           377
Premier League    380
Serie A           380
Name: match_id, dtype: int64


## Creating shot distance and shot angle features

In [9]:
# Location is currently stored as a string x, y "[100, 50]"
# To make x and y usable, we will strip the [], split by a comma, and make the data type a float

xy = (
    major_4_competitions_shots_df['location']
    .str.strip('[]')
    .str.split(',', expand=True)
    .astype(float)
)

major_4_competitions_shots_df['x'] = xy[0]
major_4_competitions_shots_df['y'] = xy[1]

print(major_4_competitions_shots_df[['location', 'x', 'y']].head())

           location      x     y
1343  [102.7, 19.9]  102.7  19.9
1344  [107.4, 40.7]  107.4  40.7
1345  [114.5, 38.1]  114.5  38.1
1346   [97.5, 44.8]   97.5  44.8
1347  [108.6, 52.9]  108.6  52.9


In [10]:
# Since x = 120 is the goal line, and y = 40 is the center of the net,
# we use the Pythagorean theorem to find the distance of the shot to the center of the goal

major_4_competitions_shots_df['shot_distance'] = np.sqrt(
   (120 - major_4_competitions_shots_df['x'])**2
   + (40 - major_4_competitions_shots_df['y'])**2
)

print(major_4_competitions_shots_df[['x', 'y', 'shot_distance']].head())

          x     y  shot_distance
1343  102.7  19.9      26.519804
1344  107.4  40.7      12.619429
1345  114.5  38.1       5.818935
1346   97.5  44.8      23.006303
1347  108.6  52.9      17.215400


In [11]:
# Measuring the shot angle using trigonometry
# shot_angle is how wide the goal looks from the shot, in degrees

# dx is the distance to the goal line (120 - x)
# dy is the side-to-side distance from the center of the goal (y - 40)

dx = 120 - major_4_competitions_shots_df['x']
dy = major_4_competitions_shots_df['y'] - 40

# The angle to each post is found with tangent (opposite / adjacent),
# and the shot angle is the difference between the far post and near post angles.
# The tangent subtraction identity combines both angles into one fraction
# tan(B - A) = (tan B - tan A) / (1 + tan B x tan A)

# 8 * dx is the goal width times the distance to the goal line,
# the comma separates the top and bottom of the fraction
# dx**2 + dy**2 - 16 squares the distances and subtracts half the goal width squared (4² = 16)

# np.arctan2 turns the fraction back into an angle
# np.degrees converts the angle from radians to degrees

major_4_competitions_shots_df['shot_angle'] = np.degrees(
   np.arctan2(8 * dx, dx**2 + dy**2 - 16)
)

print(major_4_competitions_shots_df[['x', 'y', 'shot_distance', 'shot_angle']].head())


          x     y  shot_distance  shot_angle
1343  102.7  19.9      26.519804   11.385260
1344  107.4  40.7      12.619429   35.132707
1345  114.5  38.1       5.818935   67.907319
1346   97.5  44.8      23.006303   19.324695
1347  108.6  52.9      17.215400   18.018901


In [12]:
# Testing whether shot distance and angle work, based on a penalty shot with a known distance and angle
# Penalties are taken from the penalty spot (108, 40), so distance should be about 12 coordinate units and angle about 37 degrees

penalty_shot = major_4_competitions_shots_df[major_4_competitions_shots_df['shot_type_name'] == 'Penalty'].iloc[0]

print(penalty_shot[['player_name', 'x', 'y', 'shot_distance', 'shot_angle']])

player_name      Luis Alberto Suárez Díaz
x                                   108.3
y                                    40.3
shot_distance                   11.703846
shot_angle                      37.728461
Name: 1543, dtype: object


In [13]:
# Checking if the data has reliable distance and angles based on all penalty shots

penalty_shots_df = major_4_competitions_shots_df[
    major_4_competitions_shots_df['shot_type_name'] == 'Penalty'
]

print(penalty_shots_df[['shot_distance', 'shot_angle']].describe())

       shot_distance  shot_angle
count     400.000000  400.000000
mean       11.954634   37.002736
std         0.124881    0.353756
min        11.501739   34.454976
25%        11.900420   36.869898
50%        12.000000   36.869898
75%        12.000000   37.156345
max        12.900000   38.348424


## Shot Location, Distance and Angle

### Shot Location (x and y)

StatsBomb records shot locations using an `[x, y]` coordinate system on a standardized 120 × 80 pitch. The attacking team always shoots toward x = 120, with the center of the goal at (120, 40).

When the data was saved to CSV, the coordinates were stored as text. We converted them back into numerical values and separated them into `x` and `y` columns for analysis.

### Shot Distance

Shot distance measures how far a shot is from the center of the goal at (120, 40). We used the Pythagorean theorem to calculate the straight-line distance based on the shot's x and y coordinates.

$$\text{distance} = \sqrt{(120-x)^2+(40-y)^2}$$

Distance is measured in StatsBomb coordinate units.

### Shot Angle

Shot angle measures how wide the goal appears from where a shot is taken.

The goalposts are located at (120, 36) and (120, 44). We used these positions and the shot coordinates to calculate the angle between the two goalposts.

First, we calculated the distance to the goal line (`dx`) and the distance from the center of the goal (`dy`).

$$dx = 120 - x$$

$$dy = y - 40$$

We then used the following formula to calculate the shot angle. The formula finds the angle from the shot to each post and takes the difference between them.

$$\text{angle} = \operatorname{atan2}(8\,dx,\ dx^2+dy^2-16)$$

The 8 represents the width of the goal and 16 is half the goal width squared (4²). We used `np.arctan2()` to handle angles greater than 90 degrees and `np.degrees()` to convert the result into degrees.

### Validation

We checked our calculations using penalty kicks since they are taken from a known position at (108, 40).

At this location, the expected distance is 12 coordinate units and the expected shot angle is 36.87 degrees.

Across the 400 penalties in our dataset, the median distance was 12.0 and the median angle was 36.87 degrees. Both matched the expected results, which helps confirm that our calculations are working correctly.

**References:**

* [StatsBomb Open Data](https://github.com/hudl/open-data)
* [NumPy arctan2](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html)
* [NumPy degrees](https://numpy.org/doc/stable/reference/generated/numpy.degrees.html)
* Anthropic. (2026). *Claude* (Opus 5.5 version) [Large language model]. https://claude.ai. Used to help develop and explain the shot angle calculations.

## Selecting Columns

In [14]:
# Keep only the columns needed for shot analysis

shot_columns = [
    # IDs to link shots, players, teams and matches

    'id', 'match_id', 'player_id', 'team_id',

    # Match details

    'match_date', 'competition_name', 'season_name', 'home_team', 'away_team',

    # When the shot happened

    'period', 'minute', 'second',

    # Who took the shot

    'player_name', 'position_name', 'team_name',

    # Shot location and the features created from it

    'x', 'y', 'shot_distance', 'shot_angle',

    # Shot situation

    'play_pattern_name', 'under_pressure',
    'shot_type_name', 'shot_body_part_name', 'shot_technique_name',
    'shot_first_time', 'shot_one_on_one', 'shot_open_goal',
    'shot_aerial_won', 'shot_deflected',

    # Outcome and StatsBomb's xG for comparison

    'shot_outcome_name', 'shot_statsbomb_xg',

    # To be dropped after goalkeeper, defender and teammate features are built
    
    'shot_freeze_frame'
]

shots_analysis_df = major_4_competitions_shots_df[shot_columns].copy()

print(shots_analysis_df.shape)

(37888, 32)


In [15]:
# Check how often a penalty shot is scored

print((penalty_shots_df['shot_outcome_name'] == 'Goal').mean())

0.75


In [16]:
# Remove penalties from the analysis dataset

shots_analysis_df = shots_analysis_df[shots_analysis_df['shot_type_name'] != 'Penalty'].copy()

# Check if there are any penalty shots remaining

print(shots_analysis_df.shape)
print((shots_analysis_df['shot_type_name'] == 'Penalty').sum())

(37488, 32)
0


In [17]:
# Check how often a non-penalty shot is scored

print((shots_analysis_df['shot_outcome_name'] == 'Goal').mean())

0.09520379854886897


## Penalty Shots

Penalty shots are taken from a fixed distance and angle with only the goalkeeper defending. Since our analysis focuses on what makes a good shot during attacking play, we decided to exclude penalties. In our dataset, 75% of penalty shots resulted in goals compared to 9.5% of non-penalty shots. Including penalties could influence the model's results since they represent a different type of scoring opportunity.

## Goalies, defenders and teammates in shot lane

The 'shot_freeze_frame' column contains the positions of nearby players when a shot was taken. Since this data was saved as text in the CSV, we will convert it back into a list of players. We will then use the Shapely library to count defenders and teammates inside the shooting lane. We will also check whether the goalkeeper is in the lane.

In [ ]:
# Check for missing freeze-frame data

print(shots_analysis_df['shot_freeze_frame'].isna().sum())

0


In [ ]:
# Convert the freeze-frame data from text back into Python lists

shots_analysis_df['shot_freeze_frame'] = shots_analysis_df['shot_freeze_frame'].apply(ast.literal_eval)

In [20]:
# Check the first shot's freeze frame is now a list, and how many players it has

print(type(shots_analysis_df['shot_freeze_frame'].iloc[0]))
print(len(shots_analysis_df['shot_freeze_frame'].iloc[0]))

<class 'list'>
14


In [ ]:
# Check if a player is inside the shooting lane or on its edge
# Polygon creates the shooting lane using the shooter and two goalposts

def in_shot_lane(player_x, player_y, shot_x, shot_y):
    lane = Polygon([(shot_x, shot_y), (120, 36), (120, 44)])
    return lane.covers(Point(player_x, player_y))

In [22]:
# Check whether the function works with a shooter at (100, 30): A is in the lane, B is outside, C is behind the shooter

print(in_shot_lane(110, 36, 100, 30))   # A: True
print(in_shot_lane(110, 40, 100, 30))   # B: False
print(in_shot_lane(95, 31, 100, 30))    # C: False

True
False
False


In [ ]:
# Check each player in a shot's freeze frame
# Use in_shot_lane to count defenders and teammates inside the shooting lane
# Make a True/False for goalie

def count_players_in_shot_lane(freeze_frame, shot_x, shot_y):

    # Track defenders, teammates and the goalkeeper in the shooting lane

    defenders_in_lane = 0
    teammates_in_lane = 0
    goalkeeper_in_lane = False

    for player in freeze_frame:

        # Loop through the shot freeze frame to find the coordinates, and skip if not in shot lane

        player_x, player_y = player['location']

        if in_shot_lane(player_x, player_y, shot_x, shot_y) == False:
            continue

        # Loop through to see what the player is (defender, teammate, goalie)

        if player['teammate'] == True:
            teammates_in_lane += 1
        elif player['position']['name'] == 'Goalkeeper':
            goalkeeper_in_lane = True
        else:
            defenders_in_lane += 1
    
    return defenders_in_lane, teammates_in_lane, goalkeeper_in_lane

In [ ]:
# Apply the function to each shot and save the results in three new columns

shots_analysis_df[['defenders_in_lane', 'teammates_in_lane', 'goalkeeper_in_lane']] = shots_analysis_df.apply(
    lambda row: count_players_in_shot_lane(row['shot_freeze_frame'], row['x'], row['y']),

    # Apply to every row
    axis=1,

    # Create 3 separate columns

    result_type='expand'
)

print(shots_analysis_df[['defenders_in_lane', 'teammates_in_lane', 'goalkeeper_in_lane']].head())

      defenders_in_lane  teammates_in_lane  goalkeeper_in_lane
1343                  1                  1                True
1344                  2                  0                True
1345                  1                  0                True
1346                  0                  0                True
1347                  1                  0                True


In [ ]:
# Check for any empty freeze frames

print((shots_analysis_df['shot_freeze_frame'].apply(len) == 0).sum())

0


## Players in Shooting Lane Summary

The shooting lane is the triangle between the shooter and the two goalposts at (120, 36) and (120, 44). Using the player locations in shot_freeze_frame, we used Shapely to check which players were inside this triangle and created three features:

defenders_in_lane = Number of opponents in the lane, not including the goalkeeper.

teammates_in_lane = Number of the shooter's teammates in the lane.

goalkeeper_in_lane = Whether the opposing goalkeeper is in the lane (True or False).

The goalkeeper is tracked separately from the other defenders. We also confirmed that no shots had a missing or empty freeze frame.

**References:**

* [Python ast.literal_eval](https://docs.python.org/3/library/ast.html#ast.literal_eval)
* [Shapely documentation](https://shapely.readthedocs.io/)